### STEP 1-1. 원본 데이터와 식별자 확인

In [2]:
from pathlib import Path
import pandas as pd

# 원본 파일 경로
file_path = Path("../data/raw/books_raw.xlsx")

print("현재 작업 폴더:", Path.cwd())
print("파일 존재 여부:", file_path.exists())

if file_path.exists():
    df = pd.read_excel(file_path)

    print("\n전체 행 수:", len(df))
    print("컬럼 목록:", df.columns.tolist())

    display(df[["순위", "ISBN", "상품번호", "상품명"]].head(5))

현재 작업 폴더: c:\dev\kant-axagent-study\llm-data-analysis-course\chapter13
파일 존재 여부: True

전체 행 수: 206
컬럼 목록: ['순위', 'ISBN', '상품번호', '상품명', '판매가', 'YES포인트', '저자', '출판사', '설명', '출고예상일', '관리분류']


,순위,ISBN,상품번호,상품명
0,1,9791172131784,139807878,빨강의 자서전
1,2,9791124073483,195842609,트렌드 코리아 2027
2,3,9791141691370,197068949,가호
3,4,9791198987679,197772287,가짜 돈 진짜 부
4,5,9791169666886,196557643,2027 공단기 소방단기 써니 행정법총론 기출문제집



### STEP 2-1. YES24 상세 페이지 확인

예상 1\~2분 · 네트워크 크롤링 없이 브라우저 확인

우선 첫 번째 도서의 상품번호 `139807878`로 아래 페이지에 접속해 줘.

[YES24 첫 번째 도서 상세 페이지](https://www.yes24.com/product/goods/139807878)

페이지에서 별점, 리뷰 건수, 판매지수 세 항목이 표시되는지 확인해 줘.

확인 결과를 알려주면, 이용조건을 점검한 다음 Jupyter에서 실제 수집 코드를 한 단계씩 진행하자.

### STEP 2-2. Python에서 HTML 구조 확인

예상 소요 1\~2분 · 단일 도서 테스트

우선 실제 네트워크 요청 없이 Chrome에서 HTML 원본을 확인하자.

1. 앞서 연 YES24 도서 상세 페이지에서 `Ctrl + U`
2. 새로 열린 HTML 원본 화면에서 `Ctrl + F`
3. `판매지수`, `리뷰 총점`, `회원리뷰`를 각각 검색

### STEP 2-3. 도서 1권 HTML 구조 추출

In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://www.yes24.com/product/goods/139807878"

response = requests.get(url, timeout=15)
print("HTTP 상태:", response.status_code)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    for keyword in ["리뷰 총점", "회원리뷰", "판매지수"]:
        matches = soup.find_all(
            string=lambda s: s and keyword in s
        )
        print(f"\n[{keyword}] 발견: {len(matches)}개")
        for match in matches[:2]:
            print(match.parent.get_text(" ", strip=True)[:200])

HTTP 상태: 200

[리뷰 총점] 발견: 1개
리뷰 총점

[회원리뷰] 발견: 1개
회원리뷰 (3건)

[판매지수] 발견: 2개
| 판매지수 132,366 판매지수란?
판매지수란?


### STEP 2-4. HTML 구조 확인

In [4]:
for keyword in ["리뷰 총점", "회원리뷰", "판매지수"]:
    node = soup.find(string=lambda s: s and keyword in s)

    print(f"\n[{keyword}]")

    if node:
        parent = node.parent

        for level in range(1, 4):
            print(f"\n상위 {level}단계")
            print("태그:", parent.name)
            print("class:", parent.get("class"))
            print("내용:", parent.get_text(" ", strip=True)[:300])
            parent = parent.parent


[리뷰 총점]

상위 1단계
태그: span
class: ['bgGD', 'total_rating', 'total_rating_9']
내용: 리뷰 총점

상위 2단계
태그: div
class: ['gd_rating']
내용: 리뷰 총점 8.6 / 10.0

상위 3단계
태그: div
class: ['review_starWrap']
내용: 3 명의 예스24 회원이 평가한 평균별점 리뷰 총점 8.6 / 10.0

[회원리뷰]

상위 1단계
태그: h4
class: ['tit_txt']
내용: 회원리뷰 (3건)

상위 2단계
태그: div
class: ['tm_infoSet']
내용: 회원리뷰 (3건) 매주 10건의 우수리뷰를 선정하여 YES포인트 3만원을 드립니다. 3,000원 이상 구매 후 리뷰 작성 시 YES포인트를 드립니다. (일반회원 300원, 마니아회원 600원) eBook은 다운로드 후 작성한 리뷰만 YES포인트가 지급됩니다. 클래스는 첫번째 회차 주문확정 시점부터 마지막 회차 주문확정 후 30일 이내 작성한 리뷰만 YES포인트가 지급됩니다. 사락 독서모임으로 진행된 클래스는 사락 독서모임 게시판 후기로도 노출됩니다. eBook 페이백, CD/LP, DVD/Blu-ray, 패션 및 판매금지 상품, 예

상위 3단계
태그: div
class: ['gd_infoSet', 'infoSet_noLine', 'infoSet_review', 'lazy']
내용: 회원리뷰 (3건) 매주 10건의 우수리뷰를 선정하여 YES포인트 3만원을 드립니다. 3,000원 이상 구매 후 리뷰 작성 시 YES포인트를 드립니다. (일반회원 300원, 마니아회원 600원) eBook은 다운로드 후 작성한 리뷰만 YES포인트가 지급됩니다. 클래스는 첫번째 회차 주문확정 시점부터 마지막 회차 주문확정 후 30일 이내 작성한 리뷰만 YES포인트가 지급됩니다. 사락 독서모임으로 진행된 클래스는 사락 독서모임 게시판 후기로도 노출됩니다. eBook 페이백, CD/LP, DVD/Blu-ray,

### STEP 2-5. 도서 1권 데이터 추출

In [5]:
import re

def extract(pattern, text):
    match = re.search(pattern, text)
    return match.group(1) if match else None

rating_text = soup.select_one(".gd_ratingArea").get_text(" ", strip=True)
review_text = soup.select_one(".gd_rating").get_text(" ", strip=True)
sales_text = soup.select_one(".gd_sellNum").get_text(" ", strip=True)

review_node = next(
    (x for x in soup.select("h4.tit_txt")
     if "회원리뷰" in x.get_text(" ", strip=True)),
    None
)

count_text = review_node.get_text(" ", strip=True) if review_node else ""

result = {
    "상품번호": "139807878",
    "평균별점": extract(r"평균 별점\s*([\d.]+)", rating_text),
    "리뷰총점": extract(r"리뷰 총점\s*([\d.]+)", review_text),
    "리뷰건수": extract(r"회원리뷰\s*\(([\d,]+)건\)", count_text),
    "판매지수": extract(r"판매지수\s*([\d,]+)", sales_text)
}

display(pd.DataFrame([result]))

,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
0,139807878,9.6,None,3,"132,366"


### STEP 2-6. 리뷰총점 추출 수정

In [6]:
review_node = soup.select_one(".review_starWrap .gd_rating")

if review_node:
    review_text = review_node.get_text(" ", strip=True)
    print("원본 텍스트:", review_text)

    result["리뷰총점"] = extract(
        r"리뷰 총점\s*([\d.]+)", review_text
    )

print("추출된 리뷰총점:", result["리뷰총점"])
display(pd.DataFrame([result]))

원본 텍스트: 리뷰 총점 8.6 / 10.0
추출된 리뷰총점: 8.6


,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
0,139807878,9.6,8.6,3,"132,366"


### STEP 2-7. 추가 도서 2권 추출 테스트

In [7]:
import time

test_results = []

# 2~3위 도서 테스트
for product_id in df["상품번호"].iloc[1:3].astype(str):

    url = f"https://www.yes24.com/product/goods/{product_id}"
    response = requests.get(url, timeout=15)

    print(f"상품번호 {product_id}: HTTP {response.status_code}")

    if response.status_code != 200:
        continue

    book_soup = BeautifulSoup(response.text, "html.parser")

    def get_text(selector):
        node = book_soup.select_one(selector)
        return node.get_text(" ", strip=True) if node else ""

    review_node = next(
        (x for x in book_soup.select("h4.tit_txt")
         if "회원리뷰" in x.get_text(" ", strip=True)),
        None
    )

    count_text = review_node.get_text(" ", strip=True) if review_node else ""

    test_results.append({
        "상품번호": product_id,
        "평균별점": extract(r"평균 별점\s*([\d.]+)", get_text(".gd_ratingArea")),
        "리뷰총점": extract(r"리뷰 총점\s*([\d.]+)", get_text(".review_starWrap .gd_rating")),
        "리뷰건수": extract(r"회원리뷰\s*\(([\d,]+)건\)", count_text),
        "판매지수": extract(r"판매지수\s*([\d,]+)", get_text(".gd_sellNum"))
    })

    time.sleep(2)

display(pd.DataFrame(test_results))

상품번호 195842609: HTTP 200
상품번호 197068949: HTTP 200


,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
0,195842609,9.4,9.6,47,"249,162"
1,197068949,10.0,10.0,2,"235,206"


### STEP 2-8. 도서 10권 추가 테스트

In [8]:
test_results = []

for product_id in df["상품번호"].iloc[3:13].astype(str):
    try:
        url = f"https://www.yes24.com/product/goods/{product_id}"
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        book_soup = BeautifulSoup(response.text, "html.parser")

        def get_text(selector):
            node = book_soup.select_one(selector)
            return node.get_text(" ", strip=True) if node else ""

        review_node = next(
            (x for x in book_soup.select("h4.tit_txt")
             if "회원리뷰" in x.get_text(" ", strip=True)),
            None
        )

        count_text = review_node.get_text(" ", strip=True) if review_node else ""

        test_results.append({
            "상품번호": product_id,
            "평균별점": extract(r"평균 별점\s*([\d.]+)", get_text(".gd_ratingArea")),
            "리뷰총점": extract(r"리뷰 총점\s*([\d.]+)", get_text(".review_starWrap .gd_rating")),
            "리뷰건수": extract(r"회원리뷰\s*\(([\d,]+)건\)", count_text),
            "판매지수": extract(r"판매지수\s*([\d,]+)", get_text(".gd_sellNum"))
        })

        print(product_id, "성공")

    except requests.RequestException as e:
        print(product_id, "요청 실패:", type(e).__name__)
        break

    time.sleep(2)

test_df = pd.DataFrame(test_results)
display(test_df)

197772287 성공
196557643 성공
125085536 성공
192474512 성공
257435 성공
139855099 성공
197670249 성공
8759796 성공
144255288 성공
195250585 성공


,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
0,197772287,10.0,NaN,0,"80,070"
1,196557643,9.7,9.8,22,"240,237"
2,125085536,8.9,9.0,102,"300,552"
3,192474512,8.2,8.8,106,"490,788"
4,257435,9.4,9.4,417,"445,071"
5,139855099,8.0,NaN,0,"56,478"
6,197670249,10.0,10.0,2,"51,660"
7,8759796,9.5,9.6,957,"459,831"
8,144255288,10.0,10.0,1,"41,790"
9,195250585,9.3,9.4,113,"91,875"


### STEP 3-1. YES24 도서 20권 수집 및 저장

In [9]:
import time

records = []
save_path = Path("../data/raw/books_raw_ext_partial.csv")

for i, product_id in enumerate(df["상품번호"].iloc[:20].astype(str), 1):
    try:
        url = f"https://www.yes24.com/product/goods/{product_id}"
        response = requests.get(url, timeout=15)

        if response.status_code in (403, 429):
            print("접근 제한 감지. 수집 중단:", response.status_code)
            break

        response.raise_for_status()
        book_soup = BeautifulSoup(response.text, "html.parser")

        def get_text(selector):
            node = book_soup.select_one(selector)
            return node.get_text(" ", strip=True) if node else ""

        review_node = next(
            (x for x in book_soup.select("h4.tit_txt")
             if "회원리뷰" in x.get_text(" ", strip=True)),
            None
        )
        count_text = review_node.get_text(" ", strip=True) if review_node else ""

        records.append({
            "상품번호": product_id,
            "평균별점": extract(r"평균 별점\s*([\d.]+)", get_text(".gd_ratingArea")),
            "리뷰총점": extract(r"리뷰 총점\s*([\d.]+)", get_text(".review_starWrap .gd_rating")),
            "리뷰건수": extract(r"회원리뷰\s*\(([\d,]+)건\)", count_text),
            "판매지수": extract(r"판매지수\s*([\d,]+)", get_text(".gd_sellNum"))
        })

        print(f"{i}/20 완료: {product_id}")
        time.sleep(2)

    except requests.RequestException as e:
        print("요청 실패, 수집 중단:", type(e).__name__)
        break

partial_df = pd.DataFrame(records)
partial_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("\n저장된 도서 수:", len(partial_df))
print("저장 경로:", save_path)
display(partial_df.tail())

1/20 완료: 139807878
2/20 완료: 195842609
3/20 완료: 197068949
4/20 완료: 197772287
5/20 완료: 196557643
6/20 완료: 125085536
7/20 완료: 192474512
8/20 완료: 257435
9/20 완료: 139855099
10/20 완료: 197670249
11/20 완료: 8759796
12/20 완료: 144255288
13/20 완료: 195250585
14/20 완료: 161401496
15/20 완료: 195613503
16/20 완료: 69655504
17/20 완료: 195225852
18/20 완료: 194182112
19/20 완료: 195265894
20/20 완료: 155265162

저장된 도서 수: 20
저장 경로: ..\data\raw\books_raw_ext_partial.csv


,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
15,69655504,9.4,9.4,398,"220,119"
16,195225852,9.9,10.0,21,"108,834"
17,194182112,9.6,9.8,20,"118,098"
18,195265894,9.1,9.2,27,"158,280"
19,155265162,NaN,NaN,0,"33,084"


### STEP 3-2. 전체 206권 데이터 수집

이번 코드는 다음과 같이 작동해.
- 수집한 20권은 재활용하고 나머지 186권 요청
- 도서 1권 수집할 때마다 CSV 저장
- 오류나 접근 제한이 발생하면 중단
- 중단 후 같은 셀을 실행하면 저장된 지점부터 재개

In [18]:
import re
import time
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

save_path = Path("../data/raw/books_raw_ext.csv")
partial_path = Path("../data/raw/books_raw_ext_partial.csv")

source_path = save_path if save_path.exists() else partial_path
collected = pd.read_csv(source_path, dtype={"상품번호": str})
done = set(collected["상품번호"].dropna().astype(str))

product_ids = (
    df["상품번호"].dropna().astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .tolist()
)

def get_number(page, selector, pattern):
    node = page.select_one(selector)
    if node is None:
        return None
    match = re.search(pattern, node.get_text(" ", strip=True))
    return match.group(1).replace(",", "") if match else None

for product_id in product_ids:
    if product_id in done:
        continue

    try:
        response = requests.get(
            f"https://www.yes24.com/product/goods/{product_id}",
            timeout=15
        )

        if response.status_code in (403, 429):
            print("접근 제한 감지:", response.status_code)
            break

        response.raise_for_status()
        page = BeautifulSoup(response.text, "html.parser")
        if "/Product/Base/AdultCheck/" in response.url:
            record = {
                "상품번호": product_id,
                "평균별점": None,
                "리뷰총점": None,
                "리뷰건수": None,
                "판매지수": None,
            }

            collected = pd.concat(
                [collected, pd.DataFrame([record])],
                ignore_index=True
            )
            done.add(product_id)

            temp_path = save_path.with_suffix(".tmp")
            collected.to_csv(temp_path, index=False, encoding="utf-8-sig")
            temp_path.replace(save_path)

            print(f"성인인증 필요 → 결측 저장: {product_id}")
            time.sleep(2)
            continue


        if page.select_one(".gd_sellNum") is None:
            print("페이지 구조 확인 필요:", product_id)
            break

        review_node = next(
            (n for n in page.select("h4.tit_txt")
             if "회원리뷰" in n.get_text(" ", strip=True)),
            None
        )

        review_text = (
            review_node.get_text(" ", strip=True)
            if review_node else ""
        )

        review_match = re.search(
            r"회원리뷰\s*\(([\d,]+)건\)", review_text
        )

        record = {
            "상품번호": product_id,
            "평균별점": get_number(
                page, ".gd_ratingArea", r"평균 별점\s*([\d.]+)"
            ),
            "리뷰총점": get_number(
                page, ".review_starWrap .gd_rating",
                r"리뷰 총점\s*([\d.]+)"
            ),
            "리뷰건수": (
                review_match.group(1).replace(",", "")
                if review_match else None
            ),
            "판매지수": get_number(
                page, ".gd_sellNum", r"판매지수\s*([\d,]+)"
            ),
        }

        collected = pd.concat(
            [collected, pd.DataFrame([record])],
            ignore_index=True
        )
        done.add(product_id)

        temp_path = save_path.with_suffix(".tmp")
        collected.to_csv(
            temp_path, index=False, encoding="utf-8-sig"
        )
        temp_path.replace(save_path)

        print(f"{len(done)}/{len(product_ids)} 완료: {product_id}")
        time.sleep(2)

    except requests.RequestException as exc:
        print("요청 실패:", product_id, type(exc).__name__)
        break

print("수집된 행 수:", len(collected))
print("저장 경로:", save_path)

성인인증 필요 → 결측 저장: 197995818
93/206 완료: 196518173
94/206 완료: 148032819
95/206 완료: 195108531
96/206 완료: 195804435
97/206 완료: 109884307
98/206 완료: 59463540
99/206 완료: 153497469
100/206 완료: 13137546
101/206 완료: 183809488
102/206 완료: 178054322
103/206 완료: 402246
104/206 완료: 193444436
105/206 완료: 178054374
106/206 완료: 79297023
107/206 완료: 195433349
108/206 완료: 196262204
109/206 완료: 195433343
110/206 완료: 195433329
111/206 완료: 195615644
112/206 완료: 194337979
113/206 완료: 197068167
114/206 완료: 193807963
115/206 완료: 124129609
116/206 완료: 117014613
117/206 완료: 192030311
118/206 완료: 102502665
119/206 완료: 194180493
120/206 완료: 189245464
121/206 완료: 17352
122/206 완료: 129775130
123/206 완료: 196518257
124/206 완료: 197694342
125/206 완료: 195087801
126/206 완료: 195252787
127/206 완료: 195806999
128/206 완료: 45353675
129/206 완료: 195224858
130/206 완료: 186506535
131/206 완료: 174930715
132/206 완료: 85108088
133/206 완료: 196551294
134/206 완료: 197078591
135/206 완료: 195117030
136/206 완료: 125711161
137/206 완료: 372300
138/2

### STEP 3-3. 문제 도서 HTML 진단

In [16]:
product_id = "197995818"
url = f"https://www.yes24.com/product/goods/{product_id}"

response = requests.get(url, timeout=15)
page = BeautifulSoup(response.text, "html.parser")

print("HTTP:", response.status_code)
print("최종 URL:", response.url)
print("페이지 제목:", page.title.get_text(strip=True) if page.title else "없음")

for selector in [".gd_sellNum", ".gd_ratingArea", ".gd_infoTop"]:
    node = page.select_one(selector)
    print(f"{selector}:", 
          node.get_text(" ", strip=True)[:200] if node else "없음")

print("판매지수 문구 존재:", "판매지수" in page.get_text())


HTTP: 200
최종 URL: https://www.yes24.com/Product/Base/AdultCheck/197995818?AdultCheckT=LOGIN&Message=19
페이지 제목: 없음
.gd_sellNum: 없음
.gd_ratingArea: 없음
.gd_infoTop: 없음
판매지수 문구 존재: False


### STEP 3-5. Selenium 설치

In [19]:
%pip install selenium

   ---------------------------------------- 0.0/11.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.8 MB 3.5 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/11.8 MB 3.7 MB/s eta 0:00:03
   ------- -------------------------------- 2.1/11.8 MB 3.8 MB/s eta 0:00:03
   --------- ------------------------------ 2.9/11.8 MB 3.7 MB/s eta 0:00:03
   ------------ --------------------------- 3.7/11.8 MB 4.0 MB/s eta 0:00:03
   ---------------- ----------------------- 4.7/11.8 MB 4.1 MB/s eta 0:00:02
   ------------------ --------------------- 5.5/11.8 MB 4.1 MB/s eta 0:00:02
   --------------------- ------------------ 6.3/11.8 MB 4.2 MB/s eta 0:00:02
   ------------------------ --------------- 7.3/11.8 MB 4.2 MB/s eta 0:00:02
   --------------------------- ------------ 8.1/11.8 MB 4.3 MB/s eta 0:00:01
   -------------------------------- ------- 9.4/11.8 MB 4.3 MB/s eta 0:00:01
   ----------------------------------- ---- 10.5/11.8 MB 4.4 MB/s eta 0:00:01
   --

### STEP 3-6. Selenium으로 Chrome 실행

In [20]:
from selenium import webdriver

# Selenium으로 Chrome 실행
driver = webdriver.Chrome()

# 성인인증이 필요한 도서 페이지 접속
driver.get("https://www.yes24.com/product/goods/197995818")

print("현재 URL:", driver.current_url)
print("페이지 제목:", driver.title)

UnexpectedAlertPresentException: Alert Text: [19세 이상] 나이제한 상품입니다. 로그인 후 이용하세요.
Message: unexpected alert open: {Alert text : [19세 이상] 나이제한 상품입니다. 로그인 후 이용하세요.}
  (Session info: chrome=155.0.8059.79)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff64fa25805+5ad5]
	chromedriver!(No symbol) [0x7ff64f947d50]
	chromedriver!(No symbol) [0x7ff64f7646dd]
	chromedriver!(No symbol) [0x7ff64f815f21]
	chromedriver!(No symbol) [0x7ff64f7b1d52]
	chromedriver!(No symbol) [0x7ff64f7b2b63]
	chromedriver!GetHandleVerifier [0x7ff64fe48531+428801]
	chromedriver!GetHandleVerifier [0x7ff64fe7430b+4545db]
	chromedriver!GetHandleVerifier [0x7ff64fe6823e+44850e]
	chromedriver!GetHandleVerifier [0x7ff64fb2523e+10550e]
	chromedriver!(No symbol) [0x7ff64f9557a5]
	chromedriver!(No symbol) [0x7ff64f9507e4]
	chromedriver!(No symbol) [0x7ff64f950974]
	chromedriver!(No symbol) [0x7ff64f93b5fc]
	KERNEL32!BaseThreadInitThunk [0x7ffd62f2dda7+17]
	ntdll!RtlUserThreadStart [0x7ffd6489ee4c+2c]


### STEP 3-7. 성인인증 도서 1권 추출

In [21]:
import re
from bs4 import BeautifulSoup

page = BeautifulSoup(driver.page_source, "html.parser")

def extract_selenium(selector, pattern):
    node = page.select_one(selector)
    if not node:
        return None
    match = re.search(pattern, node.get_text(" ", strip=True))
    return match.group(1) if match else None

review_node = next(
    (n for n in page.select("h4.tit_txt")
     if "회원리뷰" in n.get_text(" ", strip=True)),
    None
)

review_text = review_node.get_text(" ", strip=True) if review_node else ""

result_adult = {
    "상품번호": "197995818",
    "평균별점": extract_selenium(
        ".gd_ratingArea", r"평균 별점\s*([\d.]+)"
    ),
    "리뷰총점": extract_selenium(
        ".review_starWrap .gd_rating",
        r"리뷰 총점\s*([\d.]+)"
    ),
    "리뷰건수": (
        re.search(r"회원리뷰\s*\(([\d,]+)건\)", review_text).group(1)
        if re.search(r"회원리뷰\s*\(([\d,]+)건\)", review_text)
        else None
    ),
    "판매지수": extract_selenium(
        ".gd_sellNum", r"판매지수\s*([\d,]+)"
    )
}

print("현재 URL:", driver.current_url)
display(pd.DataFrame([result_adult]))

현재 URL: https://www.yes24.com/product/goods/197995818


,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
0,197995818,None,None,0,"18,510"


### STEP 3-8. 별점 누락 원인 확인

In [22]:
for selector in [".gd_ratingArea", ".review_starWrap", ".gd_sellNum"]:
    node = page.select_one(selector)
    print(f"\n{selector}")
    print(node.get_text(" ", strip=True)[:300] if node else "태그 없음")


.gd_ratingArea
첫번째 리뷰어가 되어주세요. | 판매지수 18,510 판매지수란? 베스트 국내도서 92위

.review_starWrap
태그 없음

.gd_sellNum
| 판매지수 18,510 판매지수란?


### STEP 3-9. 수집 누락 도서 확인

In [23]:
# 수집된 CSV 확인
ext_df = pd.read_csv(
    "../data/raw/books_raw_ext.csv",
    dtype={"상품번호": str}
)

# 상품번호 형식 통일
original_ids = df["상품번호"].astype(str).str.replace(r"\.0$", "", regex=True)
collected_ids = set(ext_df["상품번호"].astype(str).str.replace(r"\.0$", "", regex=True))

# 수집되지 않은 도서 추출
missing_books = df.loc[
    ~original_ids.isin(collected_ids),
    ["순위", "상품번호", "상품명"]
]

print("전체 도서:", len(df))
print("수집된 도서:", len(ext_df))
print("수집 누락:", len(missing_books))

display(missing_books)

전체 도서: 206
수집된 도서: 205
수집 누락: 1


,순위,상품번호,상품명
205,206,10187,공기업 강의


### STEP 3-10. 성인인증 도서 저장 상태 확인

In [24]:
display(
    ext_df.loc[
        ext_df["상품번호"] == "197995818",
        ["상품번호", "평균별점", "리뷰총점", "리뷰건수", "판매지수"]
    ]
)

,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
91,197995818,NaN,NaN,NaN,NaN


### STEP 3-11. 성인인증 도서 데이터 보완

In [25]:
from pathlib import Path
import pandas as pd

save_path = Path("../data/raw/books_raw_ext.csv")

mask = ext_df["상품번호"].astype(str) == "197995818"

assert mask.sum() == 1, "상품번호가 정확히 1개인지 확인 필요"

ext_df.loc[mask, "리뷰건수"] = 0
ext_df.loc[mask, "판매지수"] = "18,510"

# 실제 별점이 없으므로 NaN 유지
ext_df.to_csv(save_path, index=False, encoding="utf-8-sig")

display(ext_df.loc[mask])

,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
91,197995818,NaN,NaN,0.0,"18,510"


### STEP 3-12. 수집 실패 의심 도서 전체 확인

In [26]:
ext_df = pd.read_csv(
    "../data/raw/books_raw_ext.csv",
    dtype={"상품번호": str}
)

columns = ["평균별점", "리뷰총점", "리뷰건수", "판매지수"]

suspects = ext_df[ext_df[columns].isna().all(axis=1)]

print("수집 실패 의심 도서:", len(suspects))

display(
    suspects[["상품번호"]].merge(
        df[["순위", "상품번호", "상품명"]].assign(
            상품번호=lambda x: x["상품번호"].astype(str)
        ),
        on="상품번호",
        how="left"
    )
)

수집 실패 의심 도서: 0


,상품번호,순위,상품명


### STEP 3-13. 부분 수집 실패 확인

In [27]:
# 판매지수 또는 리뷰건수가 없는 도서 확인
suspects = ext_df[
    ext_df["판매지수"].isna() |
    ext_df["리뷰건수"].isna()
]

print("추가 확인 필요:", len(suspects))
display(suspects)

추가 확인 필요: 0


,상품번호,평균별점,리뷰총점,리뷰건수,판매지수


### STEP 3-14. 마지막 206번째 도서 확인

In [28]:
product_id = "10187"

driver.get(f"https://www.yes24.com/product/goods/{product_id}")
page = BeautifulSoup(driver.page_source, "html.parser")

print("최종 URL:", driver.current_url)
print("페이지 제목:", driver.title)

for selector in [".gd_sellNum", ".gd_ratingArea"]:
    node = page.select_one(selector)
    print(
        selector,
        node.get_text(" ", strip=True)[:200] if node else "없음"
    )

최종 URL: https://www.yes24.com/product/goods/10187
페이지 제목: 공기업 강의 | 이상철 | 대영문화사(임춘환) - 예스24
.gd_sellNum 없음
.gd_ratingArea 첫번째 리뷰어가 되어주세요.


### STEP 3-15. 마지막 도서의 데이터 존재 여부 확인

In [29]:
import re

text = page.get_text(" ", strip=True)

review_nodes = [
    node.get_text(" ", strip=True)
    for node in page.select("h4.tit_txt")
    if "회원리뷰" in node.get_text(" ", strip=True)
]

sales_match = re.search(r"판매지수\s*([\d,]+)", text)

print("회원리뷰:", review_nodes)
print("판매지수 숫자:", sales_match.group(1) if sales_match else None)
print("판매지수 문구 존재:", "판매지수" in text)

회원리뷰: ['회원리뷰 (0건)']
판매지수 숫자: None
판매지수 문구 존재: False


### STEP 3-16. 마지막 도서 CSV에 추가

In [30]:
save_path = Path("../data/raw/books_raw_ext.csv")

ext_df = pd.read_csv(save_path, dtype={"상품번호": str})

last_record = {
    "상품번호": "10187",
    "평균별점": None,
    "리뷰총점": None,
    "리뷰건수": 0,
    "판매지수": None
}

if "10187" not in ext_df["상품번호"].values:
    ext_df = pd.concat(
        [ext_df, pd.DataFrame([last_record])],
        ignore_index=True
    )
    ext_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("전체 수집 행 수:", len(ext_df))
display(ext_df.tail(3))

전체 수집 행 수: 206


,상품번호,평균별점,리뷰총점,리뷰건수,판매지수
203,136298166,9.4,9.4,246.0,273810
204,117527279,9.9,10.0,64.0,103170
205,10187,None,None,0.0,None


### STEP 4-1. 수집 데이터 품질 검증

In [31]:
# 상품번호를 문자열로 통일
raw_ids = df["상품번호"].astype(str).str.replace(r"\.0$", "", regex=True)
ext_ids = ext_df["상품번호"].astype(str).str.replace(r"\.0$", "", regex=True)

print("=== 데이터 검증 ===")
print("원본 도서:", len(df))
print("외부 수집:", len(ext_df))

print("\n=== 상품번호 검증 ===")
print("원본 중복:", raw_ids.duplicated().sum())
print("수집 중복:", ext_ids.duplicated().sum())
print("수집 누락:", len(set(raw_ids) - set(ext_ids)))
print("원본에 없는 상품번호:", len(set(ext_ids) - set(raw_ids)))

print("\n=== 결측치 검증 ===")
print(ext_df.isna().sum())

print("\n=== 리뷰건수 0인 도서 ===")
print((pd.to_numeric(ext_df["리뷰건수"], errors="coerce") == 0).sum())

=== 데이터 검증 ===
원본 도서: 206
외부 수집: 206

=== 상품번호 검증 ===
원본 중복: 0
수집 중복: 0
수집 누락: 0
원본에 없는 상품번호: 0

=== 결측치 검증 ===
상품번호     0
평균별점    13
리뷰총점    19
리뷰건수     0
판매지수     1
dtype: int64

=== 리뷰건수 0인 도서 ===
19


### STEP 4-2. 원본 데이터와 외부 데이터 Merge

In [32]:
from pathlib import Path
import pandas as pd

raw_path = Path("../data/raw/books_raw.xlsx")
ext_path = Path("../data/raw/books_raw_ext.csv")
total_path = Path("../data/raw/books_raw_total.csv")

# 1. 저장된 파일 다시 읽기
raw_df = pd.read_excel(raw_path, dtype={"상품번호": str})
external_df = pd.read_csv(ext_path, dtype={"상품번호": str})

# 2. 상품번호 형식 통일
for data in [raw_df, external_df]:
    data["상품번호"] = data["상품번호"].str.strip()
    data["상품번호"] = data["상품번호"].str.replace(
        r"\.0$", "", regex=True
    )

# 3. Merge 실행
total_df = raw_df.merge(
    external_df,
    on="상품번호",
    how="left",
    validate="one_to_one",
    indicator=True
)

# 4. 병합 검증
print("원본 행 수:", len(raw_df))
print("병합 행 수:", len(total_df))
print("병합 상태:")
print(total_df["_merge"].value_counts())

assert len(total_df) == len(raw_df)
assert total_df["_merge"].eq("both").all()

# 5. 저장
total_df = total_df.drop(columns="_merge")
total_df.to_csv(total_path, index=False, encoding="utf-8-sig")

print("\n저장 완료:", total_path)
print("전체 컬럼 수:", len(total_df.columns))

display(total_df.head(3))

원본 행 수: 206
병합 행 수: 206
병합 상태:
_merge
both          206
left_only       0
right_only      0
Name: count, dtype: int64

저장 완료: ..\data\raw\books_raw_total.csv
전체 컬럼 수: 15


,순위,ISBN,상품번호,상품명,판매가,YES포인트,저자,출판사,설명,출고예상일,관리분류,평균별점,리뷰총점,리뷰건수,판매지수
0,1,9791172131784,139807878,빨강의 자서전,"13,500원",750원,앤 카슨 저/민승남 역,한겨레출판,“놀라운 책이다. 몇 년간 이렇게까지 나를 흔들어놓은 글은 없었다.” _앨리스 먼로...,2,국내도서-소설/시/희곡,9.6,8.6,3.0,"132,366"
1,2,9791124073483,195842609,트렌드 코리아 2027,"19,800원","1,100원","김난도,전미영,최지혜,권정윤,한다혜,이혜원,전다현,이수진,서유현,이준영,이향은,김나...",미래의창,"무리에서 벗어나 변화에 앞장서는 ALPHA SHEEP\n\n‘알파alpha, α’는...",1,국내도서-경제 경영,9.4,9.6,47.0,"249,162"
2,3,9791141691370,197068949,가호,"16,650원",920원,무라카미 하루키 저/홍은주 역,문학동네,무라카미 하루키 3년 만의 최신 장편!\n\n‘좋은 이야기란 무엇인가’\n무라카미 ...,6,국내도서-소설/시/희곡,10.0,10.0,2.0,"235,206"


### STEP 5-1. 결측치 원인 검증

In [33]:
# 리뷰건수를 숫자로 변환 (검증용)
review_count = pd.to_numeric(
    total_df["리뷰건수"], errors="coerce"
)

# 리뷰총점 결측과 리뷰 0건 비교
print("=== 리뷰총점 ===")
print("리뷰총점 결측:", total_df["리뷰총점"].isna().sum())
print("리뷰 0건:", (review_count == 0).sum())

mismatch = total_df[
    total_df["리뷰총점"].isna() != (review_count == 0)
]
print("불일치:", len(mismatch))

# 평균별점 결측 확인
print("\n=== 평균별점 ===")
print("평균별점 결측:", total_df["평균별점"].isna().sum())

print("\n=== 판매지수 결측 도서 ===")
display(
    total_df.loc[
        total_df["판매지수"].isna(),
        ["순위", "상품명", "판매지수"]
    ]
)

=== 리뷰총점 ===
리뷰총점 결측: 19
리뷰 0건: 19
불일치: 0

=== 평균별점 ===
평균별점 결측: 13

=== 판매지수 결측 도서 ===


,순위,상품명,판매지수
205,206,공기업 강의,NaN


### STEP 5-2. 평균별점 결측 원인 확인

In [34]:
missing_rating = total_df[total_df["평균별점"].isna()]

display(
    missing_rating[
        ["순위", "상품명", "평균별점", "리뷰총점", "리뷰건수"]
    ]
)

,순위,상품명,평균별점,리뷰총점,리뷰건수
19,20,짧은 이야기들,NaN,NaN,0.0
54,55,김동희 세계관 호기심 환장 RPG 2,NaN,NaN,0.0
80,81,아메리칸 학원 1,NaN,NaN,0.0
91,92,해피투게더 1-2권 박스 세트,NaN,NaN,0.0
112,113,아메리칸 학원 2,NaN,NaN,0.0
143,144,"유리, 아이러니 그리고 신",NaN,NaN,0.0
150,151,2026 시대에듀 투자자산운용사 실제유형 모의고사 + 특별부록 PASSCODE Pr...,NaN,NaN,0.0
162,163,문학을 사랑하는 젊은이들에게,NaN,NaN,0.0
168,169,한글,NaN,NaN,0.0
188,189,꿈꾸는 이가 창조한다,NaN,NaN,0.0


### STEP 5-3. 추가 데이터 숫자형 변환

In [35]:
clean_df = total_df.copy()

numeric_columns = [
    "평균별점",
    "리뷰총점",
    "리뷰건수",
    "판매지수"
]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(
        clean_df[col].astype("string").str.replace(",", "", regex=False),
        errors="coerce"
    )

print("=== 자료형 확인 ===")
print(clean_df[numeric_columns].dtypes)

print("\n=== 결측치 확인 ===")
print(clean_df[numeric_columns].isna().sum())

display(clean_df[numeric_columns].head())

=== 자료형 확인 ===
평균별점    Float64
리뷰총점    Float64
리뷰건수    Float64
판매지수      Int64
dtype: object

=== 결측치 확인 ===
평균별점    13
리뷰총점    19
리뷰건수     0
판매지수     1
dtype: int64


,평균별점,리뷰총점,리뷰건수,판매지수
0,9.6,8.6,3.0,132366
1,9.4,9.6,47.0,249162
2,10.0,10.0,2.0,235206
3,10.0,<NA>,0.0,80070
4,9.7,9.8,22.0,240237


### STEP 5-4. 기존 데이터의 숫자 컬럼 확인

In [36]:
for col in ["판매가", "YES포인트"]:
    print(f"\n=== {col} ===")
    print("자료형:", clean_df[col].dtype)
    print("결측치:", clean_df[col].isna().sum())
    print("실제 값 예시:")
    print(clean_df[col].head(10).tolist())


=== 판매가 ===
자료형: str
결측치: 0
실제 값 예시:
['13,500원', '19,800원', '16,650원', '22,500원', '41,400원', '15,750원', '16,200원', '7,200원', '11,700원', '108,000원']

=== YES포인트 ===
자료형: str
결측치: 0
실제 값 예시:
['750원', '1,100원', '920원', '1,250원', '460원', '870원', '900원', '400원', '650원', '6,000원']


### STEP 5-5. 판매가·YES포인트 전처리

In [37]:
# 판매가, YES포인트 숫자형 변환
for col in ["판매가", "YES포인트"]:
    cleaned = (
        clean_df[col]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("원", "", regex=False)
        .str.strip()
    )

    clean_df[col] = pd.to_numeric(
        cleaned, errors="coerce"
    ).astype("Int64")

# 리뷰건수 정수형 변환
clean_df["리뷰건수"] = clean_df["리뷰건수"].astype("Int64")

# 결과 검증
columns = ["판매가", "YES포인트", "리뷰건수"]

print("=== 자료형 ===")
print(clean_df[columns].dtypes)

print("\n=== 결측치 ===")
print(clean_df[columns].isna().sum())

display(clean_df[columns].head(10))

=== 자료형 ===
판매가       Int64
YES포인트    Int64
리뷰건수      Int64
dtype: object

=== 결측치 ===
판매가       0
YES포인트    2
리뷰건수      0
dtype: int64


,판매가,YES포인트,리뷰건수
0,13500,750,3
1,19800,1100,47
2,16650,920,2
3,22500,1250,0
4,41400,460,22
5,15750,870,102
6,16200,900,106
7,7200,400,417
8,11700,650,0
9,108000,6000,2


### STEP 5-6. 변환 실패 원인 확인

In [38]:
mask = clean_df["YES포인트"].isna()

display(
    total_df.loc[
        mask,
        ["순위", "상품명", "YES포인트"]
    ]
)

,순위,상품명,YES포인트
27,28,녹스,원
205,206,공기업 강의,원


### STEP 5-7. 나머지 컬럼 상태 확인

In [39]:
# 전체 컬럼 자료형과 결측치 확인
summary = pd.DataFrame({
    "자료형": clean_df.dtypes.astype(str),
    "결측치": clean_df.isna().sum(),
    "고유값": clean_df.nunique()
})

display(summary)

,자료형,결측치,고유값
순위,int64,0,206
ISBN,int64,0,206
상품번호,str,0,206
상품명,str,0,206
판매가,Int64,0,61
YES포인트,Int64,2,63
저자,str,0,172
출판사,str,0,111
설명,str,7,196
출고예상일,int64,0,11


### STEP 5-8. ISBN과 출고예상일 검증

In [40]:
# ISBN 문자열 변환
clean_df["ISBN"] = clean_df["ISBN"].astype("string")

print("=== ISBN 자릿수 ===")
print(clean_df["ISBN"].str.len().value_counts())

print("\n=== 출고예상일 값 분포 ===")
print(clean_df["출고예상일"].value_counts().sort_index())

print("\n=== 설명 결측 도서 ===")
display(
    clean_df.loc[
        clean_df["설명"].isna(),
        ["순위", "상품명"]
    ]
)

=== ISBN 자릿수 ===
ISBN
13    206
Name: count, dtype: Int64

=== 출고예상일 값 분포 ===
출고예상일
1     188
2       7
3       1
4       1
5       1
6       2
7       1
14      1
22      1
28      2
43      1
Name: count, dtype: int64

=== 설명 결측 도서 ===


,순위,상품명
7,8,싯다르타
12,13,브레인 악셀
130,131,몬스터 캠핑장
138,139,먼작귀 2
147,148,먼작귀 3
161,162,먼작귀 4
183,184,먼작귀 5


### STEP 5-9. 최종 데이터 검증

In [41]:
checks = {
    "206행 유지": len(clean_df) == 206,
    "15개 컬럼 유지": len(clean_df.columns) == 15,
    "상품번호 중복 없음": not clean_df["상품번호"].duplicated().any(),
    "ISBN 모두 13자리": clean_df["ISBN"].str.fullmatch(r"\d{13}").all(),
    "평균별점 0~10": clean_df["평균별점"].dropna().between(0, 10).all(),
    "리뷰총점 0~10": clean_df["리뷰총점"].dropna().between(0, 10).all(),
    "리뷰건수 음수 없음": clean_df["리뷰건수"].dropna().ge(0).all(),
    "판매지수 음수 없음": clean_df["판매지수"].dropna().ge(0).all()
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\n전체 검증:", "PASS" if all(checks.values()) else "FAIL")

206행 유지: PASS
15개 컬럼 유지: PASS
상품번호 중복 없음: PASS
ISBN 모두 13자리: PASS
평균별점 0~10: PASS
리뷰총점 0~10: PASS
리뷰건수 음수 없음: PASS
판매지수 음수 없음: PASS

전체 검증: PASS


### STEP 5-10. 최종 데이터 저장

In [42]:
from pathlib import Path
import pandas as pd

# 1. 최종 저장 경로
save_path = Path("../data/processed/books_best_cleaned.csv")
save_path.parent.mkdir(parents=True, exist_ok=True)

# 2. 검증 통과 시 저장
assert all(checks.values()), "검증 실패: 저장 중단"

clean_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

# 3. 저장된 파일 다시 읽어서 검증
saved_df = pd.read_csv(
    save_path,
    dtype={"ISBN": str, "상품번호": str}
)

print("저장 경로:", save_path.resolve())
print("전체 행 수:", len(saved_df))
print("전체 컬럼 수:", len(saved_df.columns))
print("상품번호 중복:", saved_df["상품번호"].duplicated().sum())

print("\n=== 최종 결측치 ===")
print(saved_df.isna().sum())

print("\n=== 최종 저장 상태 ===")
print("PASS" if (
    len(saved_df) == 206
    and len(saved_df.columns) == 15
    and saved_df["상품번호"].is_unique
) else "FAIL")

저장 경로: C:\dev\kant-axagent-study\llm-data-analysis-course\data\processed\books_best_cleaned.csv
전체 행 수: 206
전체 컬럼 수: 15
상품번호 중복: 0

=== 최종 결측치 ===
순위         0
ISBN       0
상품번호       0
상품명        0
판매가        0
YES포인트     2
저자         0
출판사        0
설명         7
출고예상일      0
관리분류       0
평균별점      13
리뷰총점      19
리뷰건수       0
판매지수       1
dtype: int64

=== 최종 저장 상태 ===
PASS


## Chapter 13 작업 결과

최종 도서

# 206권

최종 컬럼

# 15개

병합 성공률

# 100%

최종 파일

`data/processed/books_best_cleaned.csv`

### 최종 결측치 현황

| 항목         | 결측 건수 | 처리        |
| ---------- | ----- | --------- |
| YES포인트     | 2     | 유지        |
| 설명         | 7     | 유지        |
| 평균별점       | 13    | 유지        |
| 리뷰총점       | 19    | 리뷰 0건과 일치 |
| 판매지수       | 1     | 유지        |
| 나머지 10개 컬럼 | 0     | 정상        |

### 완료한 작업

YES24 외부 데이터 수집

원본 데이터와 병합

중복·미매칭·결측 검증

숫자형 및 ISBN 자료형 정리

최종 CSV 저장 및 재검증

